# SILVER_SIH

Internações (AIH) com tipos corrigidos e nomes de negócio:

| Origem | Destino | Transformação |
|---|---|---|
| `N_AIH`, `IDENT`, `SEQ_AIH5` | `NUM_AIH`, `TIPO_AIH`, `SEQ_AIH5` | `SEQ_AIH5` vazio vira `000` |
| `ANO_CMPT` + `MES_CMPT` | `ANO_MES_CMPT` | `AAAA-MM` |
| `DT_INTER`, `DT_SAIDA` (texto) | `DT_INTERNACAO`, `DT_SAIDA` (DATE) | data inválida vira nulo (membro desconhecido na fato) |
| `MUNIC_RES` / `MUNIC_MOV` | `COD_IBGE_6_RES` / `COD_IBGE_6_ATEND` | residência do paciente / município do hospital |
| `SEXO` (1, 2, 3, 0, 9) | `SEXO` (M, F, I) | 1 = M; 2 e 3 = F; demais = I (indeterminado) |
| `IDADE` + `COD_IDADE` | `IDADE_ANOS`, `FAIXA_ETARIA` | `COD_IDADE` 4 = anos, 5 = 100 + anos, 2/3 = dias/meses (0 ano); outros = ignorada |
| `COMPLEX` | `COMPLEXIDADE` | descrição oficial (`COMPLEX.CNV`) |
| `MORTE` (0/1) | `FLG_OBITO` (S/N) | |
| `DIAS_PERM`, `VAL_TOT` | `QTD_DIAS_PERMANENCIA`, `VL_TOTAL` | nulo vira 0 (o registro é mantido) |

**Origem:** `BRONZE_SIH` + `SILVER_DOMINIO`.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO SILVER_SIH
    (NUM_AIH, TIPO_AIH, SEQ_AIH5, ANO_MES_CMPT, DT_INTERNACAO, DT_SAIDA, COD_IBGE_6_RES,
     COD_IBGE_6_ATEND, COD_CNES, COD_PROCEDIMENTO, COD_CID, COD_ESPECIALIDADE, COD_CARATER,
     SEXO, IDADE_ANOS, FAIXA_ETARIA, COMPLEXIDADE, QTD_DIAS_PERMANENCIA, VL_TOTAL, FLG_OBITO)
SELECT s.N_AIH,
       s.IDENT,
       NVL(TRIM(s.SEQ_AIH5), '000'),
       s.ANO_CMPT || '-' || s.MES_CMPT,
       TO_DATE(s.DT_INTER DEFAULT NULL ON CONVERSION ERROR, 'YYYYMMDD'),
       TO_DATE(s.DT_SAIDA DEFAULT NULL ON CONVERSION ERROR, 'YYYYMMDD'),
       s.MUNIC_RES,
       s.MUNIC_MOV,
       s.CNES,
       s.PROC_REA,
       TRIM(s.DIAG_PRINC),
       s.ESPEC,
       s.CAR_INT,
       CASE s.SEXO WHEN '1' THEN 'M' WHEN '2' THEN 'F' WHEN '3' THEN 'F' ELSE 'I' END,
       s.IDADE_ANOS,
       CASE WHEN s.IDADE_ANOS IS NULL THEN 'IGNORADA'
            WHEN s.IDADE_ANOS < 5  THEN '00-04'
            WHEN s.IDADE_ANOS < 15 THEN '05-14'
            WHEN s.IDADE_ANOS < 30 THEN '15-29'
            WHEN s.IDADE_ANOS < 45 THEN '30-44'
            WHEN s.IDADE_ANOS < 60 THEN '45-59'
            WHEN s.IDADE_ANOS < 75 THEN '60-74'
            ELSE '75+' END,
       NVL(c.DESCRICAO, 'NAO INFORMADO'),
       NVL(s.DIAS_PERM, 0),
       NVL(s.VAL_TOT, 0),
       CASE WHEN s.MORTE = 1 THEN 'S' ELSE 'N' END
FROM (
    SELECT b.*,
           CASE b.COD_IDADE WHEN '4' THEN b.IDADE
                            WHEN '5' THEN 100 + b.IDADE
                            WHEN '3' THEN 0
                            WHEN '2' THEN 0 END AS IDADE_ANOS
    FROM BRONZE_SIH b
) s
LEFT JOIN SILVER_DOMINIO c
       ON c.TABELA = 'COMPLEXIDADE'
      AND c.CODIGO = TO_CHAR(TO_NUMBER(s.COMPLEX DEFAULT NULL ON CONVERSION ERROR))
"""

In [ ]:
recarregar("SILVER_SIH", SQL, origem="BRONZE_SIH")

In [ ]:
consultar("SELECT * FROM SILVER_SIH FETCH FIRST 10 ROWS ONLY")